In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Configuration

DATA_DIR = Path(".")

MODEL_FILES = {
    "Qwen2.5-7B-Instruct":             "qwen2p5_7b_results_processed",
    "Olmo-3-7B-Instruct":      "olmo3_7b_instruct_results_processed",
    "Phi-3-mini-4k-instruct":                  "phi3_results_processed",
    "OLMo-7B-Instruct-hf":    "olmo_7b_instruct_hf_results_processed",
    "Mistral-7B-Instruct-v0.3":             "mistral7b_results_processed",
    "Llama-3.1-8B-Instruct":           "llama-3_1-8b_results_processed",
    "Hermes-2-Pro-Llama-3-8B":  "hermes2pro_llama_8b_results_processed",
}

# Column legend
# mixed  → 1=unsafe | 0=safe+invalid  (NaN never appears)
# strict → 1=unsafe | 0=safe          | NaN=invalid (nonsense / instr-misfollow)
MIXED_COL  = "original_response_mixed_binary"
STRICT_COL = "original_response_strict_binary"
LANG_COL   = "language"


# Helpers

def resolve_csv_path(file_name: str, data_dir: Path = DATA_DIR) -> Path:
    path = data_dir / file_name
    if path.exists():
        return path
    csv_path = path.with_suffix(".csv")
    if csv_path.exists():
        return csv_path
    raise FileNotFoundError(f"File not found: {file_name} or {file_name}.csv")


def to_binary(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    unexpected = numeric.dropna()[~numeric.dropna().isin([0, 1])]
    if not unexpected.empty:
        raise ValueError(f"Unexpected non-binary values: {unexpected.unique()}")
    return numeric


def safe_divide(num, den):
    return np.nan if den == 0 else num / den


# Per-model summary

def summarize_model(model_name: str, file_name: str) -> dict:
    path = resolve_csv_path(file_name)
    df   = pd.read_csv(path)

    for col in (MIXED_COL, STRICT_COL):
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in {path.name}")

    mixed  = to_binary(df[MIXED_COL])
    strict = to_binary(df[STRICT_COL])

    total_count      = len(df)                    # all rows
    unsafe_count     = int((mixed  == 1).sum())   # unsafe  (same in both schemes)
    safe_valid_count = int((strict == 0).sum())   # safe & valid only
    invalid_count    = int(strict.isna().sum())   # nonsense / instr-misfollow
    safe_total_count = total_count - unsafe_count # safe including invalids
    valid_count      = total_count - invalid_count

    return {
        "model":             model_name,
        "total_count":       total_count,
        "unsafe_count":      unsafe_count,
        "safe_total_count":  safe_total_count,   # includes invalid
        "safe_valid_count":  safe_valid_count,   # strictly safe only
        "invalid_count":     invalid_count,

        # Rates over ALL responses
        "Overall_ASR":       safe_divide(unsafe_count,     total_count),
        "Overall_SafeRate":  safe_divide(safe_total_count, total_count),

        # Rates over VALID responses only (excludes invalid/NaN)
        "Valid-response_ASR":       safe_divide(unsafe_count,     valid_count),
        "Valid-response_SafeRate":  safe_divide(safe_valid_count, valid_count),

        "invalid_rate":      safe_divide(invalid_count, total_count),
    }


# Build summary table

RATE_COLS = [
    "Overall_ASR", "Overall_SafeRate",
    "Valid-response_ASR", "Valid-response_SafeRate",
    "invalid_rate",
]

rows = [summarize_model(name, fname) for name, fname in MODEL_FILES.items()]

summary = (
    pd.DataFrame(rows)
    .sort_values("Overall_ASR", ascending=True)
    .reset_index(drop=True)
)

display_df = summary.copy()
for col in RATE_COLS:
    display_df[col] = display_df[col].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "NaN"
    )

display_df

In [ ]:
# Export for LaTeX / Overleaf

def save_summary_csv(
    df_numeric: pd.DataFrame,
    rate_cols: list[str],
    path: Path | str = "model_summary.csv",
    percent: bool = True,
) -> Path:
    out = df_numeric.copy()

    if percent:
        for col in rate_cols:
            out[col] = out[col].map(
                lambda x: f"{x:.2%}" if pd.notna(x) else "NaN"
            )

    out_path = Path(path)
    out.to_csv(out_path, index=False)
    return out_path

In [ ]:
save_summary_csv(
    df_numeric=summary,
    rate_cols=RATE_COLS,
    path="model_summary.csv",
    percent=True,
)

In [ ]:
# Overall summary (all models combined)

overall = {
    "model":             "ALL MODELS",
    "total_count":       summary["total_count"].sum(),
    "unsafe_count":      summary["unsafe_count"].sum(),
    "safe_total_count":  summary["safe_total_count"].sum(),
    "safe_valid_count":  summary["safe_valid_count"].sum(),
    "invalid_count":     summary["invalid_count"].sum(),
}

valid_count = overall["total_count"] - overall["invalid_count"]

overall["Overall_ASR"]              = safe_divide(overall["unsafe_count"],     overall["total_count"])
overall["Overall_SafeRate"]         = safe_divide(overall["safe_total_count"],  overall["total_count"])
overall["Valid-response_ASR"]       = safe_divide(overall["unsafe_count"],      valid_count)
overall["Valid-response_SafeRate"]  = safe_divide(overall["safe_valid_count"],  valid_count)
overall["invalid_rate"]             = safe_divide(overall["invalid_count"],      overall["total_count"])

overall_summary = pd.DataFrame([overall])

overall_display = overall_summary.copy()
for col in RATE_COLS:
    overall_display[col] = overall_display[col].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "NaN"
    )

overall_display

In [ ]:
save_summary_csv(
    df_numeric=overall_summary,
    rate_cols=RATE_COLS,
    path="model_Overall_summary.csv",
    percent=True,
)

In [ ]:
# Visualisations
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

PALETTE_UNSAFE  = "#e05252"
PALETTE_SAFE    = "#52a852"
PALETTE_INVALID = "#a0a0a0"
PALETTE_MODELS  = "Blues_r"

sns.set_theme(style="whitegrid", font_scale=1.15)

MODELS = summary["model"].tolist()

def save_fig(fig: plt.Figure, name: str, dpi: int = 300) -> None:
    """Save figure as both PNG and PDF into FIG_DIR."""
    for ext in ("png", "pdf"):
        out = FIG_DIR / f"{name}.{ext}"
        fig.savefig(out, dpi=dpi, bbox_inches="tight")
    print(f"Saved → {FIG_DIR / name}.[png|pdf]")


# ── 1. Overall ASR – horizontal bar (model ranking) ────────────────────────

def plot_overall_asr(df: pd.DataFrame) -> plt.Figure:
    ranked = df.sort_values("Overall_ASR", ascending=True)
    colors = sns.color_palette("Reds", len(ranked))

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(ranked["model"], ranked["Overall_ASR"], color=colors)
    ax.bar_label(bars, labels=[f"{v:.1%}" for v in ranked["Overall_ASR"]], padding=4)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_xlabel("Overall Attack Success Rate (ASR)")
    ax.set_xlim(0, ranked["Overall_ASR"].max() * 1.18)
    fig.tight_layout()
    return fig

fig1 = plot_overall_asr(summary)
save_fig(fig1, "1_overall_asr")


# ── 2. Valid-response ASR vs Overall ASR – grouped bar ─────────────────────

def plot_asr_comparison(df: pd.DataFrame) -> plt.Figure:
    ranked = df.sort_values("Overall_ASR", ascending=False)
    x      = range(len(ranked))
    width  = 0.38

    fig, ax = plt.subplots(figsize=(10, 5))
    b1 = ax.bar([i - width/2 for i in x], ranked["Overall_ASR"],
                width, label="Overall ASR",         color=PALETTE_UNSAFE,  alpha=0.85)
    b2 = ax.bar([i + width/2 for i in x], ranked["Valid-response_ASR"],
                width, label="Valid-response ASR",  color="#e09052", alpha=0.85)

    ax.bar_label(b1, labels=[f"{v:.1%}" for v in ranked["Overall_ASR"]],
                 padding=3, fontsize=9)
    ax.bar_label(b2, labels=[f"{v:.1%}" for v in ranked["Valid-response_ASR"]],
                 padding=3, fontsize=9)

    ax.set_xticks(list(x))
    ax.set_xticklabels(ranked["model"], rotation=25, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("Attack Success Rate")
    ax.legend()
    fig.tight_layout()
    return fig

fig2 = plot_asr_comparison(summary)
save_fig(fig2, "2_asr_comparison")


# ── 3. Stacked bar – response breakdown (unsafe / safe-valid / invalid) ────

def plot_response_breakdown(df: pd.DataFrame) -> plt.Figure:
    ranked = df.sort_values("Overall_ASR", ascending=False)
    total  = ranked["total_count"]

    unsafe_pct  = ranked["unsafe_count"]     / total
    valid_pct   = ranked["safe_valid_count"] / total
    invalid_pct = ranked["invalid_count"]    / total

    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(ranked))

    ax.bar(x, unsafe_pct,  label="Unsafe",       color=PALETTE_UNSAFE)
    ax.bar(x, valid_pct,   label="Safe (valid)",  color=PALETTE_SAFE,
           bottom=unsafe_pct)
    ax.bar(x, invalid_pct, label="Invalid",       color=PALETTE_INVALID,
           bottom=unsafe_pct + valid_pct)

    ax.set_xticks(list(x))
    ax.set_xticklabels(ranked["model"], rotation=25, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("Share of all responses")
    ax.legend(loc="upper right")
    fig.tight_layout()
    return fig

fig3 = plot_response_breakdown(summary)
save_fig(fig3, "3_response_breakdown")


# ── 4. Heatmap – all rates ──────────────────────────────────────────────────

def plot_rate_heatmap(df: pd.DataFrame) -> plt.Figure:
    heat_cols = ["Overall_ASR", "Overall_SafeRate",
                 "Valid-response_ASR", "Valid-response_SafeRate", "invalid_rate"]

    heat = df.set_index("model")[heat_cols].astype(float)

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.heatmap(
        heat,
        annot=True,
        fmt=".1%",
        cmap="RdYlGn_r",
        linewidths=0.5,
        ax=ax,
        cbar_kws={"format": mtick.PercentFormatter(xmax=1)},
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
    fig.tight_layout()
    return fig

fig4 = plot_rate_heatmap(summary)
save_fig(fig4, "4_rate_heatmap")


# ── 5. Safe-rate comparison (Overall vs Valid-response) ────────────────────

def plot_saferate_comparison(df: pd.DataFrame) -> plt.Figure:
    ranked = df.sort_values("Overall_SafeRate", ascending=False)
    x      = range(len(ranked))
    width  = 0.38

    fig, ax = plt.subplots(figsize=(10, 5))
    b1 = ax.bar([i - width/2 for i in x], ranked["Overall_SafeRate"],
                width, label="Overall SafeRate",        color=PALETTE_SAFE,   alpha=0.85)
    b2 = ax.bar([i + width/2 for i in x], ranked["Valid-response_SafeRate"],
                width, label="Valid-response SafeRate", color="#52c8a8", alpha=0.85)

    ax.bar_label(b1, labels=[f"{v:.1%}" for v in ranked["Overall_SafeRate"]],
                 padding=3, fontsize=9)
    ax.bar_label(b2, labels=[f"{v:.1%}" for v in ranked["Valid-response_SafeRate"]],
                 padding=3, fontsize=9)

    ax.set_xticks(list(x))
    ax.set_xticklabels(ranked["model"], rotation=25, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("Safe Rate")
    ax.legend()
    fig.tight_layout()
    return fig

fig5 = plot_saferate_comparison(summary)
save_fig(fig5, "5_saferate_comparison")


# ── 6. Invalid rate – bar ──────────────────────────────────────────────────

def plot_invalid_rate(df: pd.DataFrame) -> plt.Figure:
    ranked = df.sort_values("invalid_rate", ascending=True)
    colors = sns.color_palette("Greys", len(ranked) + 2)[2:]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(ranked["model"], ranked["invalid_rate"], color=colors)
    ax.bar_label(bars, labels=[f"{v:.1%}" for v in ranked["invalid_rate"]], padding=4)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_xlabel("Invalid / Nonsense Response Rate")
    ax.set_xlim(0, ranked["invalid_rate"].max() * 1.18)
    fig.tight_layout()
    return fig

fig6 = plot_invalid_rate(summary)
save_fig(fig6, "6_invalid_rate")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

LANG_COL   = "language"
LANG_FIG_DIR = Path("figures_language")
LANG_FIG_DIR.mkdir(exist_ok=True)

def compute_metrics_from_df(df: pd.DataFrame) -> dict:
    mixed  = to_binary(df[MIXED_COL])
    strict = to_binary(df[STRICT_COL])

    n_total          = len(df)
    unsafe_count     = int((mixed  == 1).sum())
    safe_valid_count = int((strict == 0).sum())
    invalid_count    = int(strict.isna().sum())
    safe_total_count = n_total - unsafe_count
    valid_count      = n_total - invalid_count

    return {
        "total_count":             n_total,
        "unsafe_count":            unsafe_count,
        "safe_total_count":        safe_total_count,
        "safe_valid_count":        safe_valid_count,
        "invalid_count":           invalid_count,
        "Overall_ASR":             safe_divide(unsafe_count,     n_total),
        "Overall_SafeRate":        safe_divide(safe_total_count, n_total),
        "Valid-response_ASR":      safe_divide(unsafe_count,     valid_count),
        "Valid-response_SafeRate": safe_divide(safe_valid_count, valid_count),
        "invalid_rate":            safe_divide(invalid_count,    n_total),
    }


def save_lang_fig(fig: plt.Figure, name: str, dpi: int = 300) -> None:
    for ext in ("png", "pdf"):
        fig.savefig(LANG_FIG_DIR / f"{name}.{ext}",
                    dpi=dpi, bbox_inches="tight")
    print(f"  Saved → {LANG_FIG_DIR / name}.[png|pdf]")


_frames = []
for _model_name, _file_name in MODEL_FILES.items():
    _path = resolve_csv_path(_file_name)
    _df   = pd.read_csv(_path)

    for _col in (LANG_COL, MIXED_COL, STRICT_COL):
        if _col not in _df.columns:
            raise ValueError(f"Missing column '{_col}' in {_path.name}")

    _df["_model"] = _model_name
    _frames.append(_df)

all_data = pd.concat(_frames, ignore_index=True)

# lang_summary  — ASR per language (all models combined)

lang_rows = []
for lang, grp in all_data.groupby(LANG_COL):
    lang_rows.append({"language": lang, **compute_metrics_from_df(grp)})

lang_summary = (
    pd.DataFrame(lang_rows)
    .sort_values("Overall_ASR", ascending=False)
    .reset_index(drop=True)
)

lang_display = lang_summary.copy()
for col in RATE_COLS:
    lang_display[col] = lang_display[col].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "NaN"
    )

save_summary_csv(lang_summary,  RATE_COLS, "lang_summary.csv",        percent=True)
save_summary_csv(lang_summary,  RATE_COLS, "lang_summary_nohead.csv", percent=True)
lang_display.to_csv("lang_summary_nohead.csv", index=False, header=False)

# lang_model_summary  — ASR per language × model

lang_model_rows = []
for (lang, model), grp in all_data.groupby([LANG_COL, "_model"]):
    lang_model_rows.append({
        "language": lang,
        "model":    model,
        **compute_metrics_from_df(grp)
    })

lang_model_summary = (
    pd.DataFrame(lang_model_rows)
    .sort_values(["language", "Overall_ASR"], ascending=[True, False])
    .reset_index(drop=True)
)

lang_model_display = lang_model_summary.copy()
for col in RATE_COLS:
    lang_model_display[col] = lang_model_display[col].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "NaN"
    )

save_summary_csv(lang_model_summary, RATE_COLS,
                 "lang_model_summary.csv", percent=True)
lang_model_display.to_csv("lang_model_summary_nohead.csv",
                           index=False, header=False)

# Shared plot style

sns.set_theme(style="whitegrid", font_scale=1.1)

_PALETTE_UNSAFE  = "#e05252"
_PALETTE_SAFE    = "#52a852"
_PALETTE_INVALID = "#a0a0a0"
_N_LANGS         = len(lang_summary)
_N_MODELS        = len(MODEL_FILES)


# Language-level figures

# ── L1: Overall ASR per language ────────────────────────────────────────────
def plot_lang_asr(df: pd.DataFrame) -> plt.Figure:
    ranked = df.sort_values("Overall_ASR", ascending=True)
    colors = sns.color_palette("Reds_r", len(ranked))
    fig, ax = plt.subplots(figsize=(8, max(4, _N_LANGS * 0.6)))
    bars = ax.barh(ranked["language"], ranked["Overall_ASR"], color=colors)
    ax.bar_label(bars,
                 labels=[f"{v:.1%}" for v in ranked["Overall_ASR"]],
                 padding=4)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_xlabel("Overall ASR")
    ax.set_xlim(0, ranked["Overall_ASR"].max() * 1.2)
    fig.tight_layout()
    return fig

fig_l1 = plot_lang_asr(lang_summary)
save_lang_fig(fig_l1, "L1_lang_overall_asr")


# ── L2: Overall ASR vs Valid-response ASR per language ──────────────────────
def plot_lang_asr_comparison(df: pd.DataFrame) -> plt.Figure:
    ranked = df.sort_values("Overall_ASR", ascending=False)
    x, w   = np.arange(len(ranked)), 0.38
    fig, ax = plt.subplots(figsize=(max(8, _N_LANGS * 1.2), 5))
    b1 = ax.bar(x - w/2, ranked["Overall_ASR"],
                w, label="Overall ASR",        color=_PALETTE_UNSAFE,  alpha=0.85)
    b2 = ax.bar(x + w/2, ranked["Valid-response_ASR"],
                w, label="Valid-response ASR", color="#e09052", alpha=0.85)
    ax.bar_label(b1, labels=[f"{v:.1%}" for v in ranked["Overall_ASR"]],
                 padding=3, fontsize=8)
    ax.bar_label(b2, labels=[f"{v:.1%}" for v in ranked["Valid-response_ASR"]],
                 padding=3, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(ranked["language"], rotation=30, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("ASR")
    ax.legend()
    fig.tight_layout()
    return fig

fig_l2 = plot_lang_asr_comparison(lang_summary)
save_lang_fig(fig_l2, "L2_lang_asr_comparison")


# ── L3: Stacked response breakdown per language ──────────────────────────────
def plot_lang_breakdown(df: pd.DataFrame) -> plt.Figure:
    ranked  = df.sort_values("Overall_ASR", ascending=False)
    x       = np.arange(len(ranked))
    total   = ranked["total_count"]
    up      = ranked["unsafe_count"]     / total
    vp      = ranked["safe_valid_count"] / total
    ip      = ranked["invalid_count"]    / total

    fig, ax = plt.subplots(figsize=(max(8, _N_LANGS * 1.2), 5))
    ax.bar(x, up, label="Unsafe",      color=_PALETTE_UNSAFE)
    ax.bar(x, vp, label="Safe (valid)", color=_PALETTE_SAFE,    bottom=up)
    ax.bar(x, ip, label="Invalid",      color=_PALETTE_INVALID, bottom=up + vp)
    ax.set_xticks(x)
    ax.set_xticklabels(ranked["language"], rotation=30, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("Share of responses")
    ax.legend(loc="upper right")
    fig.tight_layout()
    return fig

fig_l3 = plot_lang_breakdown(lang_summary)
save_lang_fig(fig_l3, "L3_lang_breakdown")


# ── L4: Invalid rate per language ────────────────────────────────────────────
def plot_lang_invalid(df: pd.DataFrame) -> plt.Figure:
    ranked = df.sort_values("invalid_rate", ascending=True)
    colors = sns.color_palette("Greys_r", len(ranked) + 2)[2:]
    fig, ax = plt.subplots(figsize=(8, max(4, _N_LANGS * 0.6)))
    bars = ax.barh(ranked["language"], ranked["invalid_rate"], color=colors)
    ax.bar_label(bars,
                 labels=[f"{v:.1%}" for v in ranked["invalid_rate"]],
                 padding=4)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_xlabel("Invalid rate")
    ax.set_xlim(0, ranked["invalid_rate"].max() * 1.2)
    fig.tight_layout()
    return fig

fig_l4 = plot_lang_invalid(lang_summary)
save_lang_fig(fig_l4, "L4_lang_invalid_rate")


# Language × Model figures

_pivot_overall = lang_model_summary.pivot(
    index="language", columns="model", values="Overall_ASR").astype(float)
_pivot_valid   = lang_model_summary.pivot(
    index="language", columns="model", values="Valid-response_ASR").astype(float)
_pivot_invalid = lang_model_summary.pivot(
    index="language", columns="model", values="invalid_rate").astype(float)


def _heatmap(pivot: pd.DataFrame, title: str,
             fname: str, cmap: str = "YlOrRd") -> plt.Figure:
    fig, ax = plt.subplots(
        figsize=(max(10, _N_MODELS * 1.5), max(5, _N_LANGS * 0.7))
    )
    sns.heatmap(pivot, annot=True, fmt=".1%", cmap=cmap,
                linewidths=0.4, ax=ax,
                cbar_kws={"format": mtick.PercentFormatter(xmax=1)})
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    fig.tight_layout()
    save_lang_fig(fig, fname)
    return fig

fig_lm1 = _heatmap(_pivot_overall,
                    "Overall ASR — Language × Model",
                    "LM1_heatmap_overall_asr")

fig_lm2 = _heatmap(_pivot_valid,
                    "Valid-response ASR — Language × Model",
                    "LM2_heatmap_valid_asr")

fig_lm3 = _heatmap(_pivot_invalid,
                    "Invalid Rate — Language × Model",
                    "LM3_heatmap_invalid_rate", cmap="Blues")


# ── LM4: Grouped bar — Overall ASR per language, one bar per model ───────────
def plot_grouped_bar(df: pd.DataFrame) -> plt.Figure:
    models  = sorted(df["model"].unique())
    langs   = (lang_summary
               .sort_values("Overall_ASR", ascending=False)["language"]
               .tolist())
    x       = np.arange(len(langs))
    w       = 0.8 / len(models)
    palette = sns.color_palette("tab10", len(models))

    fig, ax = plt.subplots(figsize=(max(12, len(langs) * 1.5), 6))
    for i, (model, color) in enumerate(zip(models, palette)):
        sub    = df[df["model"] == model].set_index("language")
        values = [sub.loc[l, "Overall_ASR"] if l in sub.index else np.nan
                  for l in langs]
        ax.bar(x + (i - len(models)/2 + 0.5) * w,
               values, w, label=model, color=color, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(langs, rotation=30, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("Overall ASR")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    fig.tight_layout()
    return fig

fig_lm4 = plot_grouped_bar(lang_model_summary)
save_lang_fig(fig_lm4, "LM4_grouped_bar_lang_model")


# ── LM5: Line plot — Valid-response ASR across languages per model ───────────
def plot_line_valid_asr(df: pd.DataFrame) -> plt.Figure:
    langs_ord = (lang_summary
                 .sort_values("Overall_ASR", ascending=False)["language"]
                 .tolist())
    palette = sns.color_palette("tab10", _N_MODELS)

    fig, ax = plt.subplots(figsize=(max(10, _N_LANGS * 1.2), 5))
    for model, color in zip(sorted(df["model"].unique()), palette):
        sub    = df[df["model"] == model].set_index("language")
        values = [sub.loc[l, "Valid-response_ASR"]
                  if l in sub.index else np.nan for l in langs_ord]
        ax.plot(langs_ord, values, marker="o",
                label=model, color=color, linewidth=1.8)

    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_xlabel("Language")
    ax.set_ylabel("Valid-response ASR")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    fig.tight_layout()
    return fig

fig_lm5 = plot_line_valid_asr(lang_model_summary)
save_lang_fig(fig_lm5, "LM5_line_valid_asr_per_model")

In [ ]:
RESOURCE_GROUPS = {
    "english":      "High",
    "bengali":      "Medium",
    "irish":        "Low",
    "kyrgyz":       "Low",
    "luxembourgish":"Low",
    "somali":       "Low",
    "swahili":      "Low",
}

GROUP_ORDER  = ["High", "Medium", "Low"]
GROUP_COLORS = {"High": "#4c8fcc", "Medium": "#f0a830", "Low": "#e05252"}


def assign_resource_group(language: str) -> str:
    group = RESOURCE_GROUPS.get(str(language).strip().lower(), None)
    if group is None:
        return "Unknown"
    return group


all_data["resource_group"]           = all_data[LANG_COL].apply(assign_resource_group)
lang_summary["resource_group"]       = lang_summary["language"].apply(assign_resource_group)
lang_model_summary["resource_group"] = lang_model_summary["language"].apply(assign_resource_group)


# ASR per resource group  (all models combined)
rg_rows = []
for group in GROUP_ORDER:
    grp_df = all_data[all_data["resource_group"] == group]
    if grp_df.empty:
        continue
    rg_rows.append({"resource_group": group,
                    **compute_metrics_from_df(grp_df)})

rg_summary = pd.DataFrame(rg_rows)

rg_display = rg_summary.copy()
for col in RATE_COLS:
    rg_display[col] = rg_display[col].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "NaN"
    )

save_summary_csv(rg_summary, RATE_COLS, "rg_summary.csv", percent=True)
rg_display.to_csv("rg_summary_nohead.csv", index=False, header=False)


# ASR per resource group × model

rg_model_rows = []
for (group, model), grp_df in all_data.groupby(["resource_group", "_model"]):
    rg_model_rows.append({
        "resource_group": group,
        "model":          model,
        **compute_metrics_from_df(grp_df)
    })

rg_model_summary = (
    pd.DataFrame(rg_model_rows)
    .sort_values(["resource_group", "Overall_ASR"], ascending=[True, False])
    .reset_index(drop=True)
)

rg_model_display = rg_model_summary.copy()
for col in RATE_COLS:
    rg_model_display[col] = rg_model_display[col].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "NaN"
    )

save_summary_csv(rg_model_summary, RATE_COLS,
                 "rg_model_summary.csv", percent=True)
rg_model_display.to_csv("rg_model_summary_nohead.csv",
                         index=False, header=False)


# Shared style

sns.set_theme(style="whitegrid", font_scale=1.1)

_GROUPS        = [g for g in GROUP_ORDER
                  if g in rg_summary["resource_group"].values]
_GROUP_PAL     = [GROUP_COLORS[g] for g in _GROUPS]
_MODELS_SORTED = sorted(rg_model_summary["model"].unique())
_N_MODELS      = len(_MODELS_SORTED)

_LANG_ORDER = (
    lang_summary
    .assign(
        rg_cat=pd.Categorical(
            lang_summary["resource_group"],
            categories=GROUP_ORDER, ordered=True
        )
    )
    .sort_values(["rg_cat", "Overall_ASR"], ascending=[True, False])
    ["language"]
    .tolist()
)


#  Figures

# ── RG1: Overall ASR & Valid-response ASR per resource group ─────────────────

def plot_rg_asr_grouped(df: pd.DataFrame) -> plt.Figure:
    df = df[df["resource_group"].isin(_GROUPS)].copy()
    df["resource_group"] = pd.Categorical(
        df["resource_group"], categories=_GROUPS, ordered=True)
    df = df.sort_values("resource_group")

    x, w = np.arange(len(_GROUPS)), 0.35
    fig, ax = plt.subplots(figsize=(7, 5))

    b1 = ax.bar(x - w/2, df["Overall_ASR"],
                w, label="Overall ASR",
                color=_GROUP_PAL, alpha=0.92)
    b2 = ax.bar(x + w/2, df["Valid-response_ASR"],
                w, label="Valid-response ASR",
                color=_GROUP_PAL, alpha=0.45,
                edgecolor=_GROUP_PAL, linewidth=1.5)

    ax.bar_label(b1, labels=[f"{v:.1%}" for v in df["Overall_ASR"]],
                 padding=3, fontsize=10)
    ax.bar_label(b2, labels=[f"{v:.1%}" for v in df["Valid-response_ASR"]],
                 padding=3, fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels(_GROUPS, fontsize=11)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("ASR")
    ax.legend()
    fig.tight_layout()
    return fig

fig_rg1 = plot_rg_asr_grouped(rg_summary)
save_lang_fig(fig_rg1, "RG1_rg_asr_grouped")


# ── RG2: All 5 metrics side by side per resource group ───────────────────────

def plot_rg_full_metrics(df: pd.DataFrame) -> plt.Figure:
    metrics = ["Overall_ASR", "Valid-response_ASR", "Overall_SafeRate",
               "Valid-response_SafeRate", "invalid_rate"]
    labels  = ["Overall\nASR", "Valid-resp.\nASR", "Overall\nSafe",
               "Valid-resp.\nSafe", "Invalid\nRate"]

    df = df[df["resource_group"].isin(_GROUPS)].copy()
    df["resource_group"] = pd.Categorical(
        df["resource_group"], categories=_GROUPS, ordered=True)
    df = df.sort_values("resource_group")

    x, w  = np.arange(len(metrics)), 0.22
    fig, ax = plt.subplots(figsize=(11, 5))

    for i, (group, color) in enumerate(zip(_GROUPS, _GROUP_PAL)):
        row    = df[df["resource_group"] == group].iloc[0]
        values = [row[m] for m in metrics]
        offset = (i - len(_GROUPS) / 2 + 0.5) * w
        bars   = ax.bar(x + offset, values, w,
                        label=group, color=color, alpha=0.85)
        ax.bar_label(bars, labels=[f"{v:.0%}" for v in values],
                     padding=2, fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("Rate")
    ax.legend(title="Resource group")
    fig.tight_layout()
    return fig

fig_rg2 = plot_rg_full_metrics(rg_summary)
save_lang_fig(fig_rg2, "RG2_rg_all_metrics")


# ── RG3: Stacked response breakdown per resource group ───────────────────────

def plot_rg_breakdown(df: pd.DataFrame) -> plt.Figure:
    df = df[df["resource_group"].isin(_GROUPS)].copy()
    df["resource_group"] = pd.Categorical(
        df["resource_group"], categories=_GROUPS, ordered=True)
    df = df.sort_values("resource_group")

    total = df["total_count"]
    up    = df["unsafe_count"]     / total
    vp    = df["safe_valid_count"] / total
    ip    = df["invalid_count"]    / total
    x     = np.arange(len(df))

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.bar(x, up, label="Unsafe",       color="#e05252")
    ax.bar(x, vp, label="Safe (valid)",  color="#52a852", bottom=up)
    ax.bar(x, ip, label="Invalid",       color="#a0a0a0", bottom=up + vp)

    for xi, (u, v, inv) in enumerate(zip(up, vp, ip)):
        if u   > 0.03: ax.text(xi, u/2,           f"{u:.0%}", ha="center",
                               va="center", fontsize=10, color="white", fontweight="bold")
        if v   > 0.03: ax.text(xi, u + v/2,       f"{v:.0%}", ha="center",
                               va="center", fontsize=10, color="white", fontweight="bold")
        if inv > 0.03: ax.text(xi, u + v + inv/2, f"{inv:.0%}", ha="center",
                               va="center", fontsize=10, color="white", fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(df["resource_group"].tolist(), fontsize=11)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.legend(loc="upper right")
    fig.tight_layout()
    return fig

fig_rg3 = plot_rg_breakdown(rg_summary)
save_lang_fig(fig_rg3, "RG3_rg_breakdown")


# ── RG4 / RG5 / RG6: Heatmaps  resource group × model ───────────────────────

def plot_rg_heatmap(metric: str, title: str,
                    fname: str, cmap: str = "YlOrRd") -> plt.Figure:
    pivot = (
        rg_model_summary[rg_model_summary["resource_group"].isin(_GROUPS)]
        .pivot(index="resource_group", columns="model", values=metric)
        .astype(float)
        .reindex(_GROUPS)
    )
    fig, ax = plt.subplots(
        figsize=(max(10, _N_MODELS * 1.5), len(_GROUPS) + 2))
    sns.heatmap(pivot, annot=True, fmt=".1%", cmap=cmap,
                linewidths=0.5, ax=ax,
                cbar_kws={"format": mtick.PercentFormatter(xmax=1)})
    ax.set_xlabel("")
    ax.set_ylabel("Resource group")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    fig.tight_layout()
    save_lang_fig(fig, fname)
    return fig

fig_rg4 = plot_rg_heatmap("Overall_ASR",
                            "Overall ASR — Resource Group × Model",
                            "RG4_heatmap_overall_asr")
fig_rg5 = plot_rg_heatmap("Valid-response_ASR",
                            "Valid-response ASR — Resource Group × Model",
                            "RG5_heatmap_valid_asr")
fig_rg6 = plot_rg_heatmap("invalid_rate",
                            "Invalid Rate — Resource Group × Model",
                            "RG6_heatmap_invalid_rate", cmap="Blues")


# ── RG7 / RG8: Grouped bar per model split by resource group ─────────────────

def plot_rg_per_model(df: pd.DataFrame,
                      metric: str, ylabel: str, title: str,
                      fname: str) -> plt.Figure:
    df  = df[df["resource_group"].isin(_GROUPS)].copy()
    x   = np.arange(_N_MODELS)
    w   = 0.22

    fig, ax = plt.subplots(figsize=(max(10, _N_MODELS * 1.6), 5))
    for i, (group, color) in enumerate(zip(_GROUPS, _GROUP_PAL)):
        sub    = df[df["resource_group"] == group].set_index("model")
        values = [sub.loc[m, metric] if m in sub.index else np.nan
                  for m in _MODELS_SORTED]
        offset = (i - len(_GROUPS) / 2 + 0.5) * w
        bars   = ax.bar(x + offset, values, w,
                        label=group, color=color, alpha=0.85)
        ax.bar_label(bars,
                     labels=[f"{v:.0%}" if pd.notna(v) else ""
                              for v in values],
                     padding=2, fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(_MODELS_SORTED, rotation=25, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel(ylabel)
    ax.legend(title="Resource group")
    fig.tight_layout()
    save_lang_fig(fig, fname)
    return fig

fig_rg7 = plot_rg_per_model(
    rg_model_summary,
    "Overall_ASR", "Overall ASR",
    "Overall ASR per Model — split by Resource Group",
    "RG7_overall_asr_per_model_by_rg")

fig_rg8 = plot_rg_per_model(
    rg_model_summary,
    "Valid-response_ASR", "Valid-response ASR",
    "Valid-response ASR per Model — split by Resource Group",
    "RG8_valid_asr_per_model_by_rg")


# ── RG9: Line trend High → Medium → Low per model (both ASR metrics) ─────────

def plot_rg_trend(df: pd.DataFrame) -> plt.Figure:
    df      = df[df["resource_group"].isin(_GROUPS)].copy()
    palette = sns.color_palette("tab10", _N_MODELS)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
    for ax, metric, title in zip(
        axes,
        ["Overall_ASR",        "Valid-response_ASR"],
        ["Overall ASR",        "Valid-response ASR"],
    ):
        for model, color in zip(_MODELS_SORTED, palette):
            sub    = df[df["model"] == model].set_index("resource_group")
            values = [sub.loc[g, metric] if g in sub.index else np.nan
                      for g in _GROUPS]
            ax.plot(_GROUPS, values, marker="o",
                    label=model, color=color, linewidth=1.8)

        ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
        ax.set_xlabel("Resource group")
        ax.set_ylabel(title)
        ax.grid(True, linestyle="--", alpha=0.5)

    axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    fig.tight_layout()
    return fig

fig_rg9 = plot_rg_trend(rg_model_summary)
save_lang_fig(fig_rg9, "RG9_asr_trend_by_resource_group")


# ── RG10: Per-language ASR coloured by resource group ────────────────────────

def plot_lang_asr_by_rg(df: pd.DataFrame) -> plt.Figure:
    df = df.copy()
    df["resource_group"] = pd.Categorical(
        df["resource_group"], categories=GROUP_ORDER, ordered=True)
    df = df.sort_values(["resource_group", "Overall_ASR"],
                        ascending=[True, False])

    colors = [GROUP_COLORS[g] for g in df["resource_group"]]
    x      = np.arange(len(df))
    w      = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - w/2, df["Overall_ASR"],
                w, color=colors, alpha=0.92, label="_nolegend_")
    b2 = ax.bar(x + w/2, df["Valid-response_ASR"],
                w, color=colors, alpha=0.42,
                edgecolor=colors, linewidth=1.2, label="_nolegend_")

    ax.bar_label(b1, labels=[f"{v:.0%}" for v in df["Overall_ASR"]],
                 padding=2, fontsize=8)
    ax.bar_label(b2, labels=[f"{v:.0%}" for v in df["Valid-response_ASR"]],
                 padding=2, fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(df["language"], rotation=25, ha="right", fontsize=10)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("ASR")

    from matplotlib.patches import Patch
    legend_handles = [Patch(color=GROUP_COLORS[g], label=g) for g in _GROUPS]
    ax.legend(handles=legend_handles, title="Resource group")
    fig.tight_layout()
    return fig

fig_rg10 = plot_lang_asr_by_rg(lang_summary)
save_lang_fig(fig_rg10, "RG10_lang_asr_coloured_by_rg")

In [ ]:
all_data["prompt_id"] = (
    all_data
    .groupby(["_model", LANG_COL])
    .cumcount()
    + 1
)

# MIXED  : 1=unsafe  0=safe+invalid  (never NaN)  → Overall metrics
# STRICT : 1=unsafe  0=safe          NaN=invalid  → Valid-response metrics
all_data["_mixed"]  = to_binary(all_data[MIXED_COL])
all_data["_strict"] = to_binary(all_data[STRICT_COL])

eng = (
    all_data[all_data[LANG_COL] == "english"]
    [["_model", "prompt_id", "_mixed", "_strict"]]
    .rename(columns={"_mixed":  "eng_mixed",
                     "_strict": "eng_strict"})
)

paired = (
    all_data[all_data[LANG_COL] != "english"]
    .copy()
    .merge(eng, on=["_model", "prompt_id"], how="left")
    .rename(columns={"_mixed": "lang_mixed", "_strict": "lang_strict"})
)

def compute_bypass_metrics(grp: pd.DataFrame) -> dict:
    n = len(grp)

    # ── OVERALL (mixed) ───────────────────────────────────────────────────────
    eng_safe_m   = (grp["eng_mixed"] == 0)
    eng_uns_m    = (grp["eng_mixed"] == 1)

    bypass_ov    = ((eng_safe_m) & (grp["lang_mixed"] == 1)).sum()
    protect_ov   = ((eng_uns_m)  & (grp["lang_mixed"] == 0)).sum()
    both_uns_ov  = ((eng_uns_m)  & (grp["lang_mixed"] == 1)).sum()
    both_safe_ov = ((eng_safe_m) & (grp["lang_mixed"] == 0)).sum()

    n_eng_safe_m = int(eng_safe_m.sum())
    n_eng_uns_m  = int(eng_uns_m.sum())

    asr_lang_ov  = safe_divide(int(grp["lang_mixed"].sum()), n)
    asr_eng_ov   = safe_divide(int(grp["eng_mixed"].sum()),  n)

    # ── VALID-RESPONSE (strict) ────────────────────────────────────────────────
    valid_mask   = grp["eng_strict"].notna() & grp["lang_strict"].notna()
    vp           = grp[valid_mask]
    n_valid      = len(vp)

    eng_safe_s   = (vp["eng_strict"] == 0)
    eng_uns_s    = (vp["eng_strict"] == 1)

    bypass_vr    = ((eng_safe_s) & (vp["lang_strict"] == 1)).sum()
    protect_vr   = ((eng_uns_s)  & (vp["lang_strict"] == 0)).sum()

    n_eng_safe_s = int(eng_safe_s.sum())
    n_eng_uns_s  = int(eng_uns_s.sum())

    asr_lang_vr  = safe_divide(int(vp["lang_strict"].sum()), n_valid)
    asr_eng_vr   = safe_divide(int(vp["eng_strict"].sum()),  n_valid)

    became_invalid = int((grp["eng_strict"].notna() & grp["lang_strict"].isna()).sum())

    return {
        "n_pairs":              n,
        "n_valid_pairs":        n_valid,
        "became_invalid_count": became_invalid,
        # ── Overall ──
        "bypass_count_ov":     int(bypass_ov),
        "protection_count_ov": int(protect_ov),
        "both_unsafe_ov":      int(both_uns_ov),
        "both_safe_ov":        int(both_safe_ov),
        "bypass_rate_ov":      safe_divide(bypass_ov,  n_eng_safe_m),
        "protection_rate_ov":  safe_divide(protect_ov, n_eng_uns_m),
        "net_bypass_ov":       safe_divide(bypass_ov,  n_eng_safe_m) -
                               safe_divide(protect_ov, n_eng_uns_m),
        "ASR_lang_ov":         asr_lang_ov,
        "ASR_delta_ov":        asr_lang_ov - asr_eng_ov,
        # ── Valid-response ──
        "bypass_count_vr":     int(bypass_vr),
        "protection_count_vr": int(protect_vr),
        "bypass_rate_vr":      safe_divide(bypass_vr,  n_eng_safe_s),
        "protection_rate_vr":  safe_divide(protect_vr, n_eng_uns_s),
        "net_bypass_vr":       safe_divide(bypass_vr,  n_eng_safe_s) -
                               safe_divide(protect_vr, n_eng_uns_s),
        "ASR_lang_vr":         asr_lang_vr,
        "ASR_delta_vr":        asr_lang_vr - asr_eng_vr,
    }


# Per language (all models pooled)
bypass_lang = (
    pd.DataFrame([{"language": lang, **compute_bypass_metrics(grp)}
                  for lang, grp in paired.groupby(LANG_COL)])
    .sort_values("bypass_rate_vr", ascending=False)
    .reset_index(drop=True)
)

OV_COLS = ["bypass_rate_ov","protection_rate_ov","net_bypass_ov","ASR_lang_ov","ASR_delta_ov"]
VR_COLS = ["bypass_rate_vr","protection_rate_vr","net_bypass_vr","ASR_lang_vr","ASR_delta_vr"]
ALL_RATE_COLS = OV_COLS + VR_COLS

def fmt_rate_df(df):
    out = df.copy()
    for col in ALL_RATE_COLS:
        if col not in out.columns: continue
        out[col] = out[col].map(
            lambda x: f"{x:+.2%}" if pd.notna(x) and ("delta" in col or "net" in col)
                      else (f"{x:.2%}" if pd.notna(x) else "NaN")
        )
    return out

save_summary_csv(bypass_lang, ALL_RATE_COLS, "bypass_lang_summary.csv", percent=True)

# Per language × model

bypass_lm = (
    pd.DataFrame([{"language": lang, "model": model,
                   **compute_bypass_metrics(grp)}
                  for (lang, model), grp in paired.groupby([LANG_COL, "_model"])])
    .sort_values(["language", "bypass_rate_vr"], ascending=[True, False])
    .reset_index(drop=True)
)
save_summary_csv(bypass_lm, ALL_RATE_COLS, "bypass_lang_model_summary.csv", percent=True)

# Per-prompt

prompt_bypass = (
    pd.DataFrame([{"prompt_id": pid, **compute_bypass_metrics(grp)}
                  for pid, grp in paired.groupby("prompt_id")])
    .sort_values("bypass_rate_vr", ascending=False)
    .reset_index(drop=True)
)
prompt_bypass.to_csv("prompt_bypass_summary.csv", index=False)

# Visualisations

sns.set_theme(style="whitegrid", font_scale=1.1)
_LANGS  = bypass_lang["language"].tolist()
_MODELS = sorted(bypass_lm["model"].unique())
_NM     = len(_MODELS)
C_OV    = "#e09052"
C_VR    = "#e05252"
C_PROT  = "#52a852"


# ── BP1: bypass_rate Overall vs Valid-response per language ──────────────────
def plot_bp1(df):
    x, w = np.arange(len(df)), 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - w/2, df["bypass_rate_ov"], w,
                label="Overall bypass rate", color=C_OV, alpha=0.88)
    b2 = ax.bar(x + w/2, df["bypass_rate_vr"], w,
                label="Valid-response bypass rate", color=C_VR, alpha=0.88)
    ax.bar_label(b1, labels=[f"{v:.1%}" for v in df["bypass_rate_ov"]], padding=3, fontsize=8)
    ax.bar_label(b2, labels=[f"{v:.1%}" for v in df["bypass_rate_vr"]], padding=3, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(df["language"].str.capitalize(), rotation=20, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("Bypass rate")
    ax.legend()
    fig.tight_layout()
    return fig

fig_bp1 = plot_bp1(bypass_lang)
save_lang_fig(fig_bp1, "BP1_bypass_rate_ov_vs_vr")


def plot_bp2(df):
    ranked = df.sort_values("net_bypass_vr", ascending=False)
    x, w   = np.arange(len(ranked)), 0.35

    fig, ax = plt.subplots(figsize=(9, 5))

    # ── Overall bars 
    for i, v in enumerate(ranked["net_bypass_ov"]):
        ax.bar(x[i] - w/2, v, w,
               color=C_OV,
               alpha=0.88,
               hatch="//" if v < 0 else "",
               edgecolor="white" if v >= 0 else C_OV)

    # ── Valid-response bars 
    for i, v in enumerate(ranked["net_bypass_vr"]):
        ax.bar(x[i] + w/2, v, w,
               color=C_VR,
               alpha=0.88,
               hatch="//" if v < 0 else "",
               edgecolor="white" if v >= 0 else C_VR)

    # ── labels ────────────────────────────────────────────────────────────────
    for i, v in enumerate(ranked["net_bypass_ov"]):
        ax.text(x[i] - w/2, v + (0.005 if v >= 0 else -0.005),
                f"{v:+.1%}",
                ha="center",
                va="bottom" if v >= 0 else "top",
                fontsize=8, color=C_OV)

    for i, v in enumerate(ranked["net_bypass_vr"]):
        ax.text(x[i] + w/2, v + (0.005 if v >= 0 else -0.005),
                f"{v:+.1%}",
                ha="center",
                va="bottom" if v >= 0 else "top",
                fontsize=8, color=C_VR, fontweight="bold")

    ax.axhline(0, color="black", linewidth=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(ranked["language"].str.capitalize(),
                       rotation=20, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("Net bypass gain")

    # ── legend ────────────────────────────────────────────────────────────────
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=C_OV,   alpha=0.88, label="Overall (Mixed)"),
        Patch(facecolor=C_VR,   alpha=0.88, label="Valid-response (Strict)"),
        Patch(facecolor="white", hatch="//",
              edgecolor="grey",  label="Negative = net protection"),
    ]
    ax.legend(handles=legend_elements, fontsize=9)

    fig.tight_layout()
    return fig

fig_bp2 = plot_bp2(bypass_lang)
save_lang_fig(fig_bp2, "BP2_asr_delta")

# ── BP3: ΔASR OV vs VR ───────────────────────────────────────────────────────
def plot_bp3(df):
    ranked = df.sort_values("ASR_delta_vr", ascending=False)
    x, w   = np.arange(len(ranked)), 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - w/2, ranked["ASR_delta_ov"], w,
                label="Overall ΔASR",
                color=[C_OV if v >= 0 else C_PROT for v in ranked["ASR_delta_ov"]],
                alpha=0.88)
    b2 = ax.bar(x + w/2, ranked["ASR_delta_vr"], w,
                label="Valid-response ΔASR",
                color=[C_VR if v >= 0 else C_PROT for v in ranked["ASR_delta_vr"]],
                alpha=0.88)
    ax.bar_label(b1, labels=[f"{v:+.1%}" for v in ranked["ASR_delta_ov"]], padding=3, fontsize=8)
    ax.bar_label(b2, labels=[f"{v:+.1%}" for v in ranked["ASR_delta_vr"]], padding=3, fontsize=8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(ranked["language"].str.capitalize(), rotation=20, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("ΔASR")
    ax.legend()
    fig.tight_layout()
    return fig

fig_bp3 = plot_bp3(bypass_lang)
save_lang_fig(fig_bp3, "BP3_asr_delta")


# ── BP4-BP7: heatmaps ────────────────────────────────────────────────────────
def heatmap(metric, title, fname, cmap="YlOrRd", fmt_str=".1%"):
    pivot = (
        bypass_lm
        .pivot(index="language", columns="model", values=metric)
        .astype(float).reindex(_LANGS)
    )
    fig, ax = plt.subplots(figsize=(max(10, _NM*1.5), max(4, len(_LANGS)*0.7)))
    sns.heatmap(pivot, annot=True, fmt=fmt_str, cmap=cmap, linewidths=0.4, ax=ax,
                cbar_kws={"format": mtick.PercentFormatter(xmax=1)})
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    ax.set_yticklabels([l.get_text().capitalize() for l in ax.get_yticklabels()], rotation=0)
    fig.tight_layout()
    save_lang_fig(fig, fname)
    return fig

fig_bp4 = heatmap("bypass_rate_ov",
    "Overall Bypass Rate — Language × Model\nP(lang_mixed=unsafe | eng_mixed=safe)",
    "BP4_heatmap_bypass_overall")
fig_bp5 = heatmap("bypass_rate_vr",
    "Valid-response Bypass Rate — Language × Model\nP(lang_strict=unsafe | eng_strict=safe, both valid)",
    "BP5_heatmap_bypass_valid")
fig_bp6 = heatmap("net_bypass_vr",
    "Net Bypass Gain (Valid-response) — Language × Model",
    "BP6_heatmap_net_bypass_vr", cmap="RdYlGn_r")
fig_bp7 = heatmap("ASR_delta_vr",
    "Valid-response ΔASR vs English — Language × Model",
    "BP7_heatmap_asr_delta_vr", cmap="RdYlGn_r")


# ── BP8: stacked paired outcome breakdown ────────────────────────────────────
def plot_bp8(df_paired):
    tbl = {}
    for lang, grp in df_paired.groupby(LANG_COL):
        n = len(grp)
        tbl[lang] = {
            "bypass":   ((grp["eng_mixed"]==0)&(grp["lang_mixed"]==1)).sum()/n,
            "both_uns": ((grp["eng_mixed"]==1)&(grp["lang_mixed"]==1)).sum()/n,
            "both_safe":((grp["eng_mixed"]==0)&(grp["lang_mixed"]==0)).sum()/n,
            "protect":  ((grp["eng_mixed"]==1)&(grp["lang_mixed"]==0)).sum()/n,
        }
    df_t = pd.DataFrame(tbl).T.reindex(_LANGS)
    x = np.arange(len(df_t))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x, df_t["bypass"],    label="Bypass (Eng safe→Lang unsafe)", color=C_VR)
    ax.bar(x, df_t["both_uns"],  label="Both unsafe",                   color=C_OV,
           bottom=df_t["bypass"])
    ax.bar(x, df_t["both_safe"], label="Both safe",                     color=C_PROT,
           bottom=df_t["bypass"]+df_t["both_uns"])
    ax.bar(x, df_t["protect"],   label="Protection (Eng unsafe→Lang safe)", color="#52c8a8",
           bottom=df_t["bypass"]+df_t["both_uns"]+df_t["both_safe"])
    ax.set_xticks(x)
    ax.set_xticklabels([l.capitalize() for l in _LANGS], rotation=20, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    return fig

fig_bp8 = plot_bp8(paired)
save_lang_fig(fig_bp8, "BP8_paired_outcome_breakdown")


# ── BP9: per-prompt bypass rates ─────────────────────────────────────────────
def plot_bp9(df):
    df = df.sort_values("bypass_rate_vr", ascending=True)
    x, w = np.arange(len(df)), 0.35
    fig, ax = plt.subplots(figsize=(7, max(5, len(df)*0.32)))
    ax.barh(x - w/2, df["bypass_rate_ov"], w,
            label="Overall", color=C_OV, alpha=0.85)
    ax.barh(x + w/2, df["bypass_rate_vr"], w,
            label="Valid-response", color=C_VR, alpha=0.85)
    ax.set_yticks(x)
    ax.set_yticklabels([f"Prompt {int(i)}" for i in df["prompt_id"]], fontsize=8)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_xlabel("Bypass rate across all languages & models")
    ax.legend()
    fig.tight_layout()
    return fig

fig_bp9 = plot_bp9(prompt_bypass)
save_lang_fig(fig_bp9, "BP9_prompt_bypass_rates")

In [ ]:
VALIDATORS = {
    "LlamaGuard":  "llama_guard_response_binary",
    "Nemotron":    "nemotron_response_binary",
    "WildGuard":   "wildguard_response_binary",
    "SGuard": "sguard_response_binary",
}

VAL_FIG_DIR = Path("figures_validators")
VAL_FIG_DIR.mkdir(exist_ok=True)


def save_val_fig(fig, name, dpi=300):
    for ext in ("png", "pdf"):
        fig.savefig(VAL_FIG_DIR / f"{name}.{ext}",
                    dpi=dpi, bbox_inches="tight")
    print(f"  Saved → {VAL_FIG_DIR / name}.[png|pdf]")

val_frames = []
for model_name, file_name in MODEL_FILES.items():
    path = resolve_csv_path(file_name)
    df   = pd.read_csv(path)
    df["_model"] = model_name
    val_frames.append(df)

val_data = pd.concat(val_frames, ignore_index=True)

val_data[MIXED_COL]  = to_binary(val_data[MIXED_COL])
val_data[STRICT_COL] = to_binary(val_data[STRICT_COL])
for val_col in VALIDATORS.values():
    if val_col in val_data.columns:
        val_data[val_col] = to_binary(val_data[val_col])


def compute_accuracy(df: pd.DataFrame,
                     val_col: str,
                     gold_col: str,
                     gold_is_strict: bool) -> dict:
    n_total = len(df)

    gold = df[gold_col].copy()
    val  = df[val_col].copy() if val_col in df.columns else pd.Series(
           [np.nan] * n_total, index=df.index)

    # strict: exclude NaN gold rows
    if gold_is_strict:
        valid_gold = gold.notna()
    else:
        valid_gold = pd.Series([True] * n_total, index=df.index)

    n_gold_nan = (~valid_gold).sum()

    # valid validator rows (non-NaN)
    valid_val  = val.notna()
    n_val_nan  = (~valid_val).sum()

    # comparable = both gold and validator are non-NaN
    comparable = valid_gold & valid_val
    n_comparable = comparable.sum()
    n_val_nan_on_valid_gold = (valid_gold & ~valid_val).sum()

    # agreement
    agree    = comparable & (gold == val)
    disagree = comparable & (gold != val)
    n_agree    = agree.sum()
    n_disagree = disagree.sum()

    return {
        "n_total":       n_total,
        "n_gold_nan":    int(n_gold_nan),
        "n_val_nan":     int(n_val_nan),
        "n_val_nan_vg":  int(n_val_nan_on_valid_gold),
        "n_comparable":  int(n_comparable),
        "n_agree":       int(n_agree),
        "n_disagree":    int(n_disagree),
        "accuracy":      safe_divide(n_agree, n_comparable),
        "coverage":      safe_divide(n_comparable, n_total),
    }


acc_rows = []
for val_name, val_col in VALIDATORS.items():
    row = {"validator": val_name}

    # PRIMARY: strict gold
    s = compute_accuracy(val_data, val_col, STRICT_COL, gold_is_strict=True)
    row["strict_n_comparable"] = s["n_comparable"]
    row["strict_n_agree"]      = s["n_agree"]
    row["strict_accuracy"]     = s["accuracy"]
    row["strict_coverage"]     = s["coverage"]
    row["strict_n_val_nan"]    = s["n_val_nan_vg"]

    # SECONDARY: mixed gold
    m = compute_accuracy(val_data, val_col, MIXED_COL, gold_is_strict=False)
    row["mixed_n_comparable"]  = m["n_comparable"]
    row["mixed_n_agree"]       = m["n_agree"]
    row["mixed_accuracy"]      = m["accuracy"]
    row["mixed_coverage"]      = m["coverage"]
    row["mixed_n_val_nan"]     = m["n_val_nan"]

    acc_rows.append(row)

acc_overall = pd.DataFrame(acc_rows)

ACC_RATE_COLS = ["strict_accuracy", "strict_coverage",
                 "mixed_accuracy",  "mixed_coverage"]

acc_display = acc_overall.copy()
for col in ACC_RATE_COLS:
    acc_display[col] = acc_display[col].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "NaN")

save_summary_csv(acc_overall, ACC_RATE_COLS,
                 "validator_accuracy_overall.csv", percent=True)


model_acc_rows = []
for model_name, model_df in val_data.groupby("_model"):
    for val_name, val_col in VALIDATORS.items():
        s = compute_accuracy(model_df, val_col, STRICT_COL, True)
        m = compute_accuracy(model_df, val_col, MIXED_COL,  False)
        model_acc_rows.append({
            "model":            model_name,
            "validator":        val_name,
            "strict_accuracy":  s["accuracy"],
            "strict_coverage":  s["coverage"],
            "mixed_accuracy":   m["accuracy"],
            "mixed_coverage":   m["coverage"],
        })

acc_model = pd.DataFrame(model_acc_rows)

save_summary_csv(acc_model, ACC_RATE_COLS,
                 "validator_accuracy_per_model.csv", percent=True)


sns.set_theme(style="whitegrid", font_scale=1.1)

_VALIDATORS = list(VALIDATORS.keys())
_MODELS_VAL = sorted(val_data["_model"].unique())
_NV         = len(_VALIDATORS)
_NM_VAL     = len(_MODELS_VAL)

C_STRICT = "#4c8fcc"   
C_MIXED  = "#f0a830"   


# ── VA1: Overall accuracy — Strict vs Mixed, grouped bar ─────────────────────

def plot_va1(df):
    x, w = np.arange(_NV), 0.35
    fig, ax = plt.subplots(figsize=(8, 5))

    b1 = ax.bar(x - w/2, df["strict_accuracy"], w,
                label="Strict gold (primary)",
                color=C_STRICT, alpha=0.88)
    b2 = ax.bar(x + w/2, df["mixed_accuracy"],  w,
                label="Mixed gold (secondary)",
                color=C_MIXED,  alpha=0.88)

    ax.bar_label(b1, labels=[f"{v:.1%}" for v in df["strict_accuracy"]],
                 padding=3, fontsize=9)
    ax.bar_label(b2, labels=[f"{v:.1%}" for v in df["mixed_accuracy"]],
                 padding=3, fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(df["validator"], fontsize=11)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Accuracy")
    ax.legend()
    ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.7, alpha=0.5)
    fig.tight_layout()
    return fig

fig_va1 = plot_va1(acc_overall)
save_val_fig(fig_va1, "VA1_overall_accuracy")


# ── VA2: Coverage (how often validator gave an answer) ───────────────────────

def plot_va2(df):
    x, w = np.arange(_NV), 0.35
    fig, ax = plt.subplots(figsize=(8, 4))

    b1 = ax.bar(x - w/2, df["strict_coverage"], w,
                label="Coverage (strict gold rows)",
                color=C_STRICT, alpha=0.88)
    b2 = ax.bar(x + w/2, df["mixed_coverage"],  w,
                label="Coverage (all rows)",
                color=C_MIXED,  alpha=0.88)

    ax.bar_label(b1, labels=[f"{v:.1%}" for v in df["strict_coverage"]],
                 padding=3, fontsize=9)
    ax.bar_label(b2, labels=[f"{v:.1%}" for v in df["mixed_coverage"]],
                 padding=3, fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(df["validator"], fontsize=11)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Coverage")
    ax.legend()
    fig.tight_layout()
    return fig

fig_va2 = plot_va2(acc_overall)
save_val_fig(fig_va2, "VA2_coverage")


# ── VA3: Heatmap — Strict accuracy per validator × model ─────────────────────

def plot_heatmap_acc(metric, title, fname, cmap="YlGn"):
    pivot = (
        acc_model
        .pivot(index="model", columns="validator", values=metric)
        .astype(float)
    )
    nan_mask = pivot.isna()
    fig, ax  = plt.subplots(figsize=(max(8, _NV * 1.8),
                                     max(5, _NM_VAL * 0.7)))
    sns.heatmap(pivot, annot=True, fmt=".1%", cmap=cmap,
                linewidths=0.4, ax=ax, mask=nan_mask,
                vmin=0.5, vmax=1.0,
                cbar_kws={"format": mtick.PercentFormatter(xmax=1)})

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            if nan_mask.iloc[i, j]:
                ax.text(j + 0.5, i + 0.5, "N/A",
                        ha="center", va="center",
                        fontsize=9, color="#aaaaaa", fontstyle="italic")

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    fig.tight_layout()
    save_val_fig(fig, fname)
    return fig

fig_va3 = plot_heatmap_acc(
    "strict_accuracy",
    "Accuracy vs Strict Gold — Validator × Model",
    "VA3_heatmap_strict_accuracy")

fig_va4 = plot_heatmap_acc(
    "mixed_accuracy",
    "Accuracy vs Mixed Gold — Validator × Model",
    "VA4_heatmap_mixed_accuracy",
    cmap="YlOrRd")


# ── VA5: Per-model accuracy line plot ─────────────────────────────────────────

def plot_va5(df, metric, title, fname):
    palette = sns.color_palette("tab10", _NV)
    fig, ax  = plt.subplots(figsize=(max(10, _NM_VAL * 1.3), 5))

    for val_name, color in zip(_VALIDATORS, palette):
        sub    = df[df["validator"] == val_name].set_index("model")
        values = [sub.loc[m, metric] if m in sub.index else np.nan
                  for m in _MODELS_VAL]
        ax.plot(_MODELS_VAL, values, marker="o",
                label=val_name, color=color, linewidth=1.8)

    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylim(0.4, 1.05)
    ax.set_xlabel("")
    ax.set_ylabel("Accuracy")
    ax.tick_params(axis="x", rotation=25)
    ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.7, alpha=0.5)
    ax.legend(title="Validator", bbox_to_anchor=(1.01, 1),
              loc="upper left", fontsize=9)
    fig.tight_layout()
    save_val_fig(fig, fname)
    return fig

fig_va5 = plot_va5(acc_model,
                   "strict_accuracy",
                   "Strict Gold Accuracy per Model — Validator Comparison",
                   "VA5_line_strict_accuracy_per_model")

fig_va6 = plot_va5(acc_model,
                   "mixed_accuracy",
                   "Mixed Gold Accuracy per Model — Validator Comparison",
                   "VA6_line_mixed_accuracy_per_model")

In [ ]:
def compute_fnr_fpr(df: pd.DataFrame,
                    val_col: str,
                    gold_col: str,
                    gold_is_strict: bool) -> dict:
    n_total = len(df)

    gold = df[gold_col].copy()
    val  = (df[val_col].copy() if val_col in df.columns
            else pd.Series([np.nan] * n_total, index=df.index))

    # strict: exclude NaN gold rows
    valid_gold = gold.notna() if gold_is_strict else pd.Series(
                 [True] * n_total, index=df.index)

    # comparable: both non-NaN
    comparable = valid_gold & val.notna()

    g = gold[comparable]
    v = val[comparable]
    n_comparable = int(comparable.sum())

    TP = int(((g == 1) & (v == 1)).sum())
    FN = int(((g == 1) & (v == 0)).sum())
    FP = int(((g == 0) & (v == 1)).sum())
    TN = int(((g == 0) & (v == 0)).sum())

    n_actual_unsafe = TP + FN
    n_actual_safe   = FP + TN

    return {
        "n_comparable":   n_comparable,
        "TP": TP, "FN": FN, "FP": FP, "TN": TN,
        "n_actual_unsafe": n_actual_unsafe,
        "n_actual_safe":   n_actual_safe,
        "FNR": safe_divide(FN, n_actual_unsafe),
        "FPR": safe_divide(FP, n_actual_safe),
        "TPR": safe_divide(TP, n_actual_unsafe),   
        "TNR": safe_divide(TN, n_actual_safe),      
    }


fnr_rows = []
for val_name, val_col in VALIDATORS.items():
    row = {"validator": val_name}

    s = compute_fnr_fpr(val_data, val_col, STRICT_COL, True)
    m = compute_fnr_fpr(val_data, val_col, MIXED_COL,  False)

    for prefix, res in [("strict_", s), ("mixed_", m)]:
        for k, v in res.items():
            row[prefix + k] = v

    fnr_rows.append(row)

fnr_overall = pd.DataFrame(fnr_rows)

FNR_RATE_COLS = ["strict_FNR", "strict_FPR", "strict_TPR", "strict_TNR",
                 "mixed_FNR",  "mixed_FPR",  "mixed_TPR",  "mixed_TNR"]

fnr_display = fnr_overall.copy()
for col in FNR_RATE_COLS:
    fnr_display[col] = fnr_display[col].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "NaN")

save_summary_csv(fnr_overall, FNR_RATE_COLS,
                 "validator_fnr_fpr_overall.csv", percent=True)

# Fix FNR_RATE_COLS — only columns that exist in BOTH dataframes ────────
FNR_RATE_COLS = ["strict_FNR", "strict_FPR", "strict_TPR", "strict_TNR",
                 "mixed_FNR",  "mixed_FPR"]    # removed mixed_TPR, mixed_TNR


# Fix per-model row construction — add missing mixed_TPR / TNR ──────────
model_fnr_rows = []
for model_name, model_df in val_data.groupby("_model"):
    for val_name, val_col in VALIDATORS.items():
        s = compute_fnr_fpr(model_df, val_col, STRICT_COL, True)
        m = compute_fnr_fpr(model_df, val_col, MIXED_COL,  False)
        model_fnr_rows.append({
            "model":       model_name,
            "validator":   val_name,
            # strict
            "strict_FNR":  s["FNR"], "strict_FPR": s["FPR"],
            "strict_TPR":  s["TPR"], "strict_TNR": s["TNR"],
            "strict_TP":   s["TP"],  "strict_FN":  s["FN"],
            "strict_FP":   s["FP"],  "strict_TN":  s["TN"],
            # mixed — include TPR and TNR now
            "mixed_FNR":   m["FNR"], "mixed_FPR":  m["FPR"],
            "mixed_TPR":   m["TPR"], "mixed_TNR":  m["TNR"],
            "mixed_TP":    m["TP"],  "mixed_FN":   m["FN"],
            "mixed_FP":    m["FP"],  "mixed_TN":   m["TN"],
        })

fnr_model = pd.DataFrame(model_fnr_rows)

# Fix FNR_RATE_COLS for per-model save — all columns now present ─────────
FNR_RATE_COLS_MODEL = ["strict_FNR", "strict_FPR", "strict_TPR", "strict_TNR",
                       "mixed_FNR",  "mixed_FPR",  "mixed_TPR",  "mixed_TNR"]

save_summary_csv(fnr_model, FNR_RATE_COLS_MODEL,
                 "validator_fnr_fpr_per_model.csv", percent=True)


# Continue with visualisations ───────────────────────────────
C_FNR  = "#e05252"
C_FPR  = "#4c8fcc"

_VALIDATORS_LIST = list(VALIDATORS.keys())
_NV = len(_VALIDATORS_LIST)


def plot_vb1(df):
    x, w = np.arange(_NV), 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    b1 = ax.bar(x - w/2, df["strict_FNR"], w, color=C_FNR, alpha=0.88,
                label="FNR — missed unsafe\n(gold=1, validator=0)")
    b2 = ax.bar(x + w/2, df["strict_FPR"], w, color=C_FPR, alpha=0.88,
                label="FPR — false alarm\n(gold=0, validator=1)")
    ax.bar_label(b1, labels=[f"{v:.1%}" for v in df["strict_FNR"]],
                 padding=3, fontsize=9)
    ax.bar_label(b2, labels=[f"{v:.1%}" for v in df["strict_FPR"]],
                 padding=3, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(df["validator"], fontsize=11)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylim(0, max(df["strict_FNR"].max(), df["strict_FPR"].max()) * 1.25)
    ax.set_ylabel("Error rate")
    ax.legend(fontsize=9)
    fig.tight_layout()
    return fig

fig_vb1 = plot_vb1(fnr_overall)
save_val_fig(fig_vb1, "VB1_fnr_fpr_strict")


def plot_vb2(df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
    for ax, gold, title in zip(
        axes,
        ["strict", "mixed"],
        ["Strict Gold", "Mixed Gold"]
    ):
        x, w = np.arange(_NV), 0.35
        b1 = ax.bar(x - w/2, df[f"{gold}_FNR"], w, color=C_FNR, alpha=0.88,
                    label="FNR (missed unsafe)")
        b2 = ax.bar(x + w/2, df[f"{gold}_FPR"], w, color=C_FPR, alpha=0.88,
                    label="FPR (false alarm)")
        ax.bar_label(b1, labels=[f"{v:.1%}" for v in df[f"{gold}_FNR"]],
                     padding=3, fontsize=8.5)
        ax.bar_label(b2, labels=[f"{v:.1%}" for v in df[f"{gold}_FPR"]],
                     padding=3, fontsize=8.5)
        ax.set_xticks(x)
        ax.set_xticklabels(df["validator"], fontsize=10)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
        ax.set_ylabel("Error rate")
        ax.legend(fontsize=8)
    fig.tight_layout()
    return fig

fig_vb2 = plot_vb2(fnr_overall)
save_val_fig(fig_vb2, "VB2_fnr_fpr_strict_vs_mixed")


def plot_heatmap_err(metric, title, fname, cmap="YlOrRd"):
    pivot    = (fnr_model
                .pivot(index="model", columns="validator", values=metric)
                .astype(float))
    nan_mask = pivot.isna()
    fig, ax  = plt.subplots(figsize=(max(8, _NV * 1.8),
                                      max(5, len(pivot) * 0.7)))
    sns.heatmap(pivot, annot=True, fmt=".1%", cmap=cmap,
                linewidths=0.4, ax=ax, mask=nan_mask,
                vmin=0.0, vmax=1.0,
                cbar_kws={"format": mtick.PercentFormatter(xmax=1)})
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            if nan_mask.iloc[i, j]:
                ax.text(j+0.5, i+0.5, "N/A", ha="center", va="center",
                        fontsize=9, color="#aaaaaa", fontstyle="italic")
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    fig.tight_layout()
    save_val_fig(fig, fname)
    return fig

fig_vb3 = plot_heatmap_err("strict_FNR",
    "False Negative Rate (Strict Gold) — Model × Validator",
    "VB3_heatmap_fnr_strict")

fig_vb4 = plot_heatmap_err("strict_FPR",
    "False Positive Rate (Strict Gold) — Model × Validator",
    "VB4_heatmap_fpr_strict", cmap="Blues")

fig_vb5 = plot_heatmap_err("mixed_FNR",
    "False Negative Rate (Mixed Gold) — Model × Validator",
    "VB5_heatmap_fnr_mixed")

fig_vb6 = plot_heatmap_err("mixed_FPR",
    "False Positive Rate (Mixed Gold) — Model × Validator",
    "VB6_heatmap_fpr_mixed", cmap="Blues")


def plot_vb7(df):
    palette = sns.color_palette("tab10", _NV)
    fig, ax  = plt.subplots(figsize=(7, 6))
    for (_, row), color in zip(df.iterrows(), palette):
        ax.scatter(row["strict_FPR"], row["strict_FNR"],
                   s=180, color=color, zorder=5, label=row["validator"])
        ax.annotate(row["validator"],
                    (row["strict_FPR"], row["strict_FNR"]),
                    textcoords="offset points", xytext=(8, 4), fontsize=10)
    ax.scatter([0], [0], marker="*", s=250, color="gold",
               edgecolors="black", zorder=6, label="Ideal (FNR=0, FPR=0)")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_xlabel("FPR  (false alarm rate on safe responses)", fontsize=11)
    ax.set_ylabel("FNR  (miss rate on unsafe responses)", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.4)
    fig.tight_layout()
    return fig

fig_vb7 = plot_vb7(fnr_overall)
save_val_fig(fig_vb7, "VB7_scatter_fnr_fpr_tradeoff")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

try:
    VAL_FIG_DIR
except NameError:
    VAL_FIG_DIR = Path("figures_validators")
    VAL_FIG_DIR.mkdir(exist_ok=True)
    def save_val_fig(fig, name, dpi=300):
        for ext in ("png", "pdf"):
            fig.savefig(VAL_FIG_DIR / f"{name}.{ext}", dpi=dpi, bbox_inches="tight")
        print(f"  Saved → {VAL_FIG_DIR / name}.[png|pdf]")

acc = pd.read_csv("validator_lang_accuracy.csv")  

def pct_to_float(s):
    if pd.isna(s): return np.nan
    if isinstance(s, str): return float(s.replace("%", "")) / 100
    return float(s)

for col in ["strict_accuracy","strict_coverage","mixed_accuracy","mixed_coverage"]:
    acc[col] = acc[col].apply(pct_to_float)

sns.set_theme(style="whitegrid", font_scale=1.05)

_LANGS = sorted(acc["language"].unique())
_VALS  = sorted(acc["validator"].unique())
_NL    = len(_LANGS)
_NV    = len(_VALS)

C_STRICT = "#4c8fcc"
C_MIXED  = "#f0a830"


def heatmap(metric, title, fname, cmap="YlGn", vmin=0.0, vmax=1.0):
    pivot    = (acc.pivot(index="language", columns="validator", values=metric)
                  .astype(float).reindex(_LANGS))
    nan_mask = pivot.isna()
    fig, ax  = plt.subplots(figsize=(max(8, _NV*1.8), max(4, _NL*0.65)))
    sns.heatmap(pivot, annot=True, fmt=".1%", cmap=cmap,
                linewidths=0.4, ax=ax, mask=nan_mask, vmin=vmin, vmax=vmax,
                cbar_kws={"format": mtick.PercentFormatter(xmax=1)})
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            if nan_mask.iloc[i, j]:
                ax.text(j+0.5, i+0.5, "N/A", ha="center", va="center",
                        fontsize=9, color="#aaa", fontstyle="italic")
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
    ax.set_yticklabels([l.get_text().capitalize()
                        for l in ax.get_yticklabels()], rotation=0)
    fig.tight_layout()
    save_val_fig(fig, fname)
    return fig


# ── VLA1: heatmap accuracy Strict ────────────────────────────────────────────
heatmap("strict_accuracy",
        "Accuracy vs Strict Gold — Language × Validator",
        "VLA1_heatmap_accuracy_strict", cmap="YlGn", vmin=0.5)

# ── VLA2: heatmap accuracy Mixed ─────────────────────────────────────────────
heatmap("mixed_accuracy",
        "Accuracy vs Mixed Gold — Language × Validator",
        "VLA2_heatmap_accuracy_mixed", cmap="YlGn", vmin=0.5)

# ── VLA3: heatmap coverage Strict ────────────────────────────────────────────
heatmap("strict_coverage",
        "Coverage (Strict Gold) — Language × Validator\n"
        "proportion of responses with valid gold AND validator label",
        "VLA3_heatmap_coverage_strict", cmap="Blues", vmin=0.0)


# ── VLA4: grouped bar — accuracy Strict vs Mixed, per language ───────────────
#         one figure with subplots per validator
def plot_acc_grouped(df):
    fig, axes = plt.subplots(1, _NV, figsize=(_NV * 3.4, 5), sharey=True)
    if _NV == 1: axes = [axes]

    for ax, val_name in zip(axes, _VALS):
        sub = df[df["validator"] == val_name].set_index("language").reindex(_LANGS)
        x, w = np.arange(_NL), 0.38
        b1 = ax.bar(x - w/2, sub["strict_accuracy"], w,
                    color=C_STRICT, alpha=0.88, label="Strict")
        b2 = ax.bar(x + w/2, sub["mixed_accuracy"],  w,
                    color=C_MIXED,  alpha=0.88, label="Mixed")
        ax.set_xticks(x)
        ax.set_xticklabels([l.capitalize() for l in _LANGS],
                           rotation=90, fontsize=8)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
        ax.set_ylim(0, 1.05)

    axes[0].set_ylabel("Accuracy")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2,
               fontsize=9, bbox_to_anchor=(0.5, 1.04))
    fig.tight_layout()
    save_val_fig(fig, "VLA4_accuracy_strict_vs_mixed_per_validator")
    return fig

plot_acc_grouped(acc)


# ── VLA5: grouped bar — accuracy per language, one bar per validator ─────────
#         (Strict gold)
def plot_acc_by_language(df, metric, title, fname):
    palette = sns.color_palette("tab10", _NV)
    ranked  = (df.groupby("language")[metric].mean()
                 .sort_values(ascending=False).index.tolist())
    x, w = np.arange(_NL), 0.18
    fig, ax = plt.subplots(figsize=(max(11, _NL*1.5), 5))
    for i, (val_name, color) in enumerate(zip(_VALS, palette)):
        sub    = df[df["validator"] == val_name].set_index("language")
        values = [sub.loc[l, metric] if l in sub.index else np.nan for l in ranked]
        offset = (i - _NV/2 + 0.5) * w
        ax.bar(x + offset, values, w, label=val_name, color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([l.capitalize() for l in ranked], rotation=20, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Accuracy")
    ax.legend(title="Validator", fontsize=9)
    fig.tight_layout()
    save_val_fig(fig, fname)
    return fig

plot_acc_by_language(acc, "strict_accuracy",
    "Strict Gold Accuracy per Language — Validator Comparison\n"
    "(sorted by mean accuracy descending)",
    "VLA5_accuracy_per_language_strict")


# ── VLA6: coverage per language (single bar per language — same for all) ─────
def plot_coverage(df):
    cov = (df.groupby("language")["strict_coverage"].mean()
             .sort_values(ascending=False))
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = sns.color_palette("Blues_r", len(cov))
    bars = ax.barh([l.capitalize() for l in cov.index], cov.values, color=colors)
    ax.bar_label(bars, labels=[f"{v:.1%}" for v in cov.values], padding=4)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_xlabel("Strict gold coverage")
    ax.set_xlim(0, 1.05)
    ax.invert_yaxis()
    fig.tight_layout()
    save_val_fig(fig, "VLA6_coverage_per_language")
    return fig

plot_coverage(acc)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

try:
    VAL_FIG_DIR
except NameError:
    VAL_FIG_DIR = Path("figures_validators")
    VAL_FIG_DIR.mkdir(exist_ok=True)
    def save_val_fig(fig, name, dpi=300):
        for ext in ("png", "pdf"):
            fig.savefig(VAL_FIG_DIR / f"{name}.{ext}", dpi=dpi, bbox_inches="tight")
        print(f"  Saved → {VAL_FIG_DIR / name}.[png|pdf]")

def pct(s):
    if pd.isna(s): return np.nan
    if isinstance(s, str): return float(s.replace("%","")) / 100
    return float(s)

sns.set_theme(style="whitegrid", font_scale=1.05)

C_FNR = "#e05252"
C_FPR = "#4c8fcc"

lang = pd.read_csv("validator_lang_fnr_fpr.csv")
for c in ["FNR", "strict_FPR", "mixed_FPR"]:
    lang[c] = lang[c].apply(pct)

_LANGS = sorted(lang["language"].unique())
_VALS  = sorted(lang["validator"].unique())
_NL    = len(_LANGS)
_NV    = len(_VALS)


def heatmap(metric, title, fname, cmap):
    pivot    = (lang.pivot(index="language", columns="validator", values=metric)
                  .astype(float).reindex(_LANGS))
    nan_mask = pivot.isna()
    fig, ax  = plt.subplots(figsize=(max(8, _NV*1.8), max(4, _NL*0.65)))
    sns.heatmap(pivot, annot=True, fmt=".1%", cmap=cmap, linewidths=0.4,
                ax=ax, mask=nan_mask, vmin=0, vmax=1,
                cbar_kws={"format": mtick.PercentFormatter(xmax=1)})
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            if nan_mask.iloc[i, j]:
                ax.text(j+0.5, i+0.5, "N/A", ha="center", va="center",
                        fontsize=9, color="#aaa", fontstyle="italic")
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
    ax.set_yticklabels([l.get_text().capitalize()
                        for l in ax.get_yticklabels()], rotation=0)
    fig.tight_layout()
    save_val_fig(fig, fname)
    return fig


# ── VE-L1: heatmap FNR ────────────────────────────────────────────────────────
heatmap("FNR",
    "FNR per Language × Validator\nP(validator=safe | gold=unsafe) — lower is better",
    "VEL1_heatmap_fnr_language", "YlOrRd")

# ── VE-L2: heatmap FPR (Strict) ──────────────────────────────────────────────
heatmap("strict_FPR",
    "FPR (Strict Gold) per Language × Validator\nP(validator=unsafe | gold=safe) — lower is better",
    "VEL2_heatmap_fpr_strict_language", "Blues")

# ── VE-L3: heatmap FPR (Mixed) ───────────────────────────────────────────────
heatmap("mixed_FPR",
    "FPR (Mixed Gold) per Language × Validator\nP(validator=unsafe | gold=safe) — lower is better",
    "VEL3_heatmap_fpr_mixed_language", "Blues")


# ── VE-L4: grouped bar FNR per language ──────────────────────────────────────
def grouped_bar(metric, title, fname, ylab):
    palette = sns.color_palette("tab10", _NV)
    items   = (lang.groupby("language")[metric].mean()
                 .sort_values(ascending=False).index.tolist())
    x, w = np.arange(_NL), 0.18
    fig, ax = plt.subplots(figsize=(max(11, _NL*1.5), 5))
    for i, (val_name, color) in enumerate(zip(_VALS, palette)):
        sub    = lang[lang["validator"] == val_name].set_index("language")
        values = [sub.loc[it, metric] if it in sub.index else np.nan
                  for it in items]
        offset = (i - _NV/2 + 0.5) * w
        ax.bar(x + offset, values, w, label=val_name, color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([it.capitalize() for it in items], rotation=20, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(ylab)
    ax.legend(title="Validator", fontsize=9)
    fig.tight_layout()
    save_val_fig(fig, fname)
    return fig

grouped_bar("FNR",
    "FNR per Language — Validator Comparison\n(sorted by mean FNR descending)",
    "VEL4_bar_fnr_language", "FNR")

grouped_bar("strict_FPR",
    "FPR (Strict) per Language — Validator Comparison\n(sorted by mean FPR descending)",
    "VEL5_bar_fpr_language", "FPR (Strict)")


# ── VE-L6: FNR vs FPR scatter trade-off, one panel per validator ─────────────
def scatter_tradeoff():
    palette    = sns.color_palette("tab10", _NL)
    lang_color = dict(zip(_LANGS, palette))

    fig, axes = plt.subplots(1, _NV, figsize=(_NV * 3.6, 4.2),
                             sharex=True, sharey=True)
    if _NV == 1: axes = [axes]

    for ax, val_name in zip(axes, _VALS):
        sub = lang[lang["validator"] == val_name]
        for _, r in sub.iterrows():
            ax.scatter(r["strict_FPR"], r["FNR"],
                       s=130, color=lang_color[r["language"]],
                       zorder=5, edgecolors="white")
            ax.annotate(r["language"].capitalize()[:3],
                        (r["strict_FPR"], r["FNR"]),
                        textcoords="offset points", xytext=(5, 3), fontsize=7)
        ax.scatter([0],[0], marker="*", s=190, color="gold",
                   edgecolors="black", zorder=6)
        ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
        ax.set_xlabel("FPR")
        ax.grid(True, linestyle="--", alpha=0.4)

    axes[0].set_ylabel("FNR")
    fig.suptitle("FNR vs FPR Trade-off per Language\n"
                 "(gold star = ideal point; each marker = one language)",
                 fontweight="bold", y=1.08)
    fig.tight_layout()
    save_val_fig(fig, "VEL6_scatter_fnr_fpr_language")
    return fig

scatter_tradeoff()

In [ ]:
LANGUAGES = sorted(val_data[LANG_COL].unique())
val_lang_rows = []

for lang in LANGUAGES:
    lang_df = val_data[val_data[LANG_COL] == lang]

    for val_name, val_col in VALIDATORS.items():

        # ── accuracy ─────────────────────────────────────────────────────────
        def acc(df, col, gold_col, strict):
            gold = df[gold_col].copy()
            val  = df[col].copy() if col in df.columns else \
                   pd.Series([np.nan]*len(df), index=df.index)
            if strict:
                mask = gold.notna() & val.notna()
            else:
                mask = val.notna()
            g, v = gold[mask], val[mask]
            n = int(mask.sum())
            if n == 0: return np.nan, np.nan
            agree = int((g == v).sum())
            return safe_divide(agree, n), safe_divide(n, len(df))

        s_acc, s_cov = acc(lang_df, val_col, STRICT_COL, True)
        m_acc, m_cov = acc(lang_df, val_col, MIXED_COL,  False)

        # ── FNR / FPR ─────────────────────────────────────────────────────────
        s_err = compute_fnr_fpr(lang_df, val_col, STRICT_COL, True)
        m_err = compute_fnr_fpr(lang_df, val_col, MIXED_COL,  False)

        val_lang_rows.append({
            "language":  lang,
            "validator": val_name,

            # counts
            "strict_n_comparable": s_err["n_comparable"],
            "mixed_n_comparable":  m_err["n_comparable"],

            # accuracy
            "strict_accuracy": s_acc,
            "mixed_accuracy":  m_acc,
            "strict_coverage": s_cov,
            "mixed_coverage":  m_cov,

            # FNR / FPR
            "FNR":        s_err["FNR"],   
            "strict_FPR": s_err["FPR"],
            "mixed_FPR":  m_err["FPR"],

            # confusion matrix (strict)
            "TP": s_err["TP"], "FN": s_err["FN"],
            "FP": s_err["FP"], "TN": s_err["TN"],
            "n_actual_unsafe": s_err["n_actual_unsafe"],
            "n_actual_safe":   s_err["n_actual_safe"],
        })

val_lang = pd.DataFrame(val_lang_rows)

RATE_COLS_VL = ["strict_accuracy", "mixed_accuracy",
                "strict_coverage", "mixed_coverage",
                "FNR", "strict_FPR", "mixed_FPR"]

save_summary_csv(val_lang, RATE_COLS_VL,
                 "validator_lang_summary.csv", percent=True)



sns.set_theme(style="whitegrid", font_scale=1.05)

_LANGS      = sorted(val_lang["language"].unique())
_VAL_NAMES  = list(VALIDATORS.keys())
_NL         = len(_LANGS)
_NV         = len(_VAL_NAMES)

C_STRICT = "#4c8fcc"
C_MIXED  = "#f0a830"
C_FNR    = "#e05252"
C_FPR    = "#4c8fcc"


def heatmap_vl(metric, title, fname,
               cmap="YlOrRd", vmin=0.0, vmax=1.0,
               fmt_str=".1%", center=None):
    """Heatmap: rows=language, cols=validator."""
    pivot    = (val_lang
                .pivot(index="language", columns="validator", values=metric)
                .astype(float)
                .reindex(_LANGS))
    nan_mask = pivot.isna()

    fig, ax  = plt.subplots(figsize=(max(8, _NV * 1.8),
                                      max(4, _NL * 0.65)))
    kwargs = dict(annot=True, fmt=fmt_str, cmap=cmap,
                  linewidths=0.4, ax=ax, mask=nan_mask,
                  cbar_kws={"format": (mtick.PercentFormatter(xmax=1)
                                       if "%" in fmt_str else None)})
    if center is not None:
        kwargs["center"] = center
    else:
        kwargs.update({"vmin": vmin, "vmax": vmax})

    sns.heatmap(pivot, **kwargs)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            if nan_mask.iloc[i, j]:
                ax.text(j+0.5, i+0.5, "N/A", ha="center", va="center",
                        fontsize=9, color="#aaaaaa", fontstyle="italic")

    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
    ax.set_yticklabels([l.get_text().capitalize()
                        for l in ax.get_yticklabels()], rotation=0)
    fig.tight_layout()
    save_val_fig(fig, fname)
    return fig


# ── VL1: Accuracy (Strict) — language × validator ────────────────────────────
fig_vl1 = heatmap_vl("strict_accuracy",
    "Accuracy vs Strict Gold — Language × Validator",
    "VL1_heatmap_accuracy_strict", cmap="YlGn")

# ── VL2: Accuracy (Mixed) — language × validator ─────────────────────────────
fig_vl2 = heatmap_vl("mixed_accuracy",
    "Accuracy vs Mixed Gold — Language × Validator",
    "VL2_heatmap_accuracy_mixed", cmap="YlGn")

# ── VL3: FNR — language × validator ──────────────────────────────────────────
fig_vl3 = heatmap_vl("FNR",
    "False Negative Rate — Language × Validator",
    "VL3_heatmap_fnr", cmap="YlOrRd")

# ── VL4: FPR Strict — language × validator ───────────────────────────────────
fig_vl4 = heatmap_vl("strict_FPR",
    "False Positive Rate (Strict Gold) — Language × Validator",
    "VL4_heatmap_fpr_strict", cmap="Blues")

# ── VL6: Grouped bar — FNR per language, one bar per validator ───────────────
def plot_vl6(df):
    palette = sns.color_palette("tab10", _NV)
    ranked  = (df.groupby("language")["FNR"]
                 .mean()
                 .sort_values(ascending=False)
                 .index.tolist())

    x, w = np.arange(_NL), 0.18
    fig, ax = plt.subplots(figsize=(max(10, _NL * 1.4), 5))

    for i, (val_name, color) in enumerate(zip(_VAL_NAMES, palette)):
        sub    = df[df["validator"] == val_name].set_index("language")
        values = [sub.loc[l, "FNR"] if l in sub.index else np.nan
                  for l in ranked]
        offset = (i - _NV/2 + 0.5) * w
        ax.bar(x + offset, values, w,
               label=val_name, color=color, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([l.capitalize() for l in ranked],
                       rotation=20, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_ylabel("FNR")
    ax.legend(title="Validator", fontsize=9)
    fig.tight_layout()
    return fig

fig_vl6 = plot_vl6(val_lang)
save_val_fig(fig_vl6, "VL6_bar_fnr_per_language")



# ── VL8: Coverage (Strict) — how often validator could classify ───────────────
fig_vl8 = heatmap_vl("strict_coverage",
    "Strict Gold Coverage — Language × Validator",
    "VL8_heatmap_coverage_strict", cmap="Blues")


# ── TABLE 1: Accuracy per language × validator ────────────────────────────────
acc_lang = val_lang[[
    "language", "validator",
    "strict_n_comparable", "mixed_n_comparable",
    "strict_accuracy", "strict_coverage",
    "mixed_accuracy",  "mixed_coverage",
]].copy()

save_summary_csv(acc_lang,
                 ["strict_accuracy","strict_coverage",
                  "mixed_accuracy", "mixed_coverage"],
                 "validator_lang_accuracy.csv", percent=True)


# ── TABLE 2: FNR / FPR per language × validator ───────────────────────────────
fnr_lang = val_lang[[
    "language", "validator",
    "n_actual_unsafe", "n_actual_safe",
    "TP", "FN", "FP", "TN",
    "FNR", "strict_FPR", "mixed_FPR",
]].copy()

save_summary_csv(fnr_lang,
                 ["FNR", "strict_FPR", "mixed_FPR"],
                 "validator_lang_fnr_fpr.csv", percent=True)